In [0]:
dim_incident_type = spark.sql(f"select * from regis_healthcare.silver.incidents;")
dim_incident_type.createOrReplaceTempView("incidents")

In [0]:
# 6. Dim_Incident_Type -- >Source: incidents
# | Column            |
# | ----------------- |
# | incident_type_key |
# | incident_type     |
# | severity          |

from pyspark.sql.functions import col,when
dim_incident_type = dim_incident_type.withColumn("incident_type_key",when(col("incident_type")=="AGGRESSION",1)
     .when(col("incident_type")=="BEHAVIOURAL ISSUE",2)
     .when(col("incident_type")=="CHOKING",3)
     .when(col("incident_type")=="ELOPEMENT",4)
     .when(col("incident_type")=="EQUIPMENT FAILURE",5)
     .when(col("incident_type")=="FALL",6)
     .when(col("incident_type")=="INFECTION",7)
     .when(col("incident_type")=="MEDICATION ERROR",8)
     .when(col("incident_type")=="OTHER",9)
     .when(col("incident_type")=="PRESSURE INJURY",10)
     .when(col("incident_type")=="SKIN TEAR",11)
     .when(col("incident_type")=="UNKNOWN",12)
     .when(col("incident_type")=="WANDERING",13).otherwise(0).cast("int"))


dim_incident_type = dim_incident_type.select(
     "incident_type_key",
    "incident_type",
    "severity" 
)
display(dim_incident_type)


#### cataloge 

In [0]:
dim_incident_type.write\
    .format("delta")\
    .option("mergeSchema","true")\
    .option("overwriteSchema","true")\
        .option("delta.enableChangeDataFeed","true")\
            .mode("overwrite")\
.saveAsTable(f"regis_healthcare.gold.dim_incident_type")

In [0]:
dim_incident_type.write\
    .format("delta")\
    .option("mergeSchema","true")\
    .option("overwriteSchema","true")\
        .option("delta.enableChangeDataFeed","true")\
            .mode("overwrite")\
.saveAsTable(f"regis_healthcare.gold.sb_dim_incident_type")
print(dim_incident_type.count())

In [0]:
# from delta.tables import DeltaTable
# from pyspark.sql import functions as F

# # Load Delta table with correct fully-qualified name
# delta_table = DeltaTable.forName(spark, "regis_healthcare.gold.dim_incident_type")
# # Create DataFrame from source table with correct fully-qualified name
# # sb_dim_products
# df_child_products = (
#     spark.table("regis_healthcare.gold.sb_dim_incident_type")
#     .select("*")
# )
# # Perform merge
# delta_table.alias("target").merge(
#     source=df_child_products.alias("source"),
#     condition="target.incident_type_key = source.incident_type_key"
# ).whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()

In [0]:
dim_df = spark.sql(f"select * from regis_healthcare.gold.dim_incident_type;")
print(dim_df.count())

sb_dim_df = spark.sql(f"select * from regis_healthcare.gold.sb_dim_incident_type;")
print(sb_dim_df.count())

#### s3 loading

In [0]:
# gold load to s3
dim_incident_type.write\
    .format("delta")\
    .option("mergeSchema","true")\
    .option("overwriteSchema","true")\
        .option("delta.enableChangeDataFeed","true")\
            .mode("overwrite")\
.save(f"s3://regis-healthcare/gold-delta-table/dim_incident_type")

In [0]:
# from delta.tables import DeltaTable

# # ✅ Path to your Delta table stored in S3
# delta_table_path = f"s3://regis-healthcare/gold-delta-table/dim_incident_type"

# # ✅ Load target Delta table
# delta_table = DeltaTable.forPath(spark, delta_table_path)

# # ✅ Source DataFrame (example: df_child_products)
# source_df = dim_incident_type

# # ✅ Perform MERGE with upsert logic
# (
#     delta_table.alias("target")
#     .merge(
#         source_df.alias("source"),
#         "target.incident_type_key = source.incident_type_key"
#     )
#     .whenMatchedUpdateAll()      # Update all columns when matched
#     .whenNotMatchedInsertAll()   # Insert all columns when not matched
#     .execute()
# )
